In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Sri_Aurobindo_Marg_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,383.0,175.0,155.0,111.0,158.0,184.0,98.0,45.0,118.0,115.0,286.0,264.0
1,2,367.0,244.0,121.0,132.0,158.0,151.0,110.0,57.0,197.0,123.0,244.0,267.0
2,3,343.0,190.0,126.0,122.0,203.0,139.0,108.0,63.0,NaN,121.0,279.0,216.0
3,4,376.0,239.0,124.0,140.0,238.0,188.0,61.0,62.0,61.0,112.0,281.0,154.0
4,5,355.0,186.0,133.0,147.0,193.0,228.0,87.0,48.0,55.0,108.0,282.0,171.0
5,6,346.0,127.0,114.0,135.0,180.0,167.0,58.0,50.0,70.0,98.0,231.0,257.0
6,7,335.0,144.0,133.0,133.0,229.0,234.0,52.0,50.0,62.0,94.0,245.0,217.0
7,8,342.0,157.0,119.0,127.0,196.0,252.0,50.0,49.0,72.0,111.0,240.0,264.0
8,9,351.0,112.0,122.0,152.0,159.0,149.0,83.0,48.0,86.0,117.0,228.0,162.0
9,10,302.0,250.0,140.0,161.0,154.0,133.0,103.0,55.0,107.0,115.0,204.0,225.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,383.0,175.000000,155.000000,111.000000,158.000000,184.000000,98.000000,45.000000,118.000000,115.000000,286.000000,264.000000
1,2,367.0,244.000000,121.000000,132.000000,158.000000,151.000000,110.000000,57.000000,77.424242,123.000000,244.000000,267.000000
2,3,343.0,190.000000,126.000000,122.000000,203.000000,139.000000,108.000000,63.000000,77.424242,121.000000,279.000000,216.000000
3,4,376.0,239.000000,124.000000,140.000000,238.000000,188.000000,61.000000,62.000000,61.000000,112.000000,281.000000,154.000000
4,5,355.0,186.000000,133.000000,147.000000,193.000000,228.000000,87.000000,48.000000,55.000000,108.000000,282.000000,171.000000
5,6,346.0,127.000000,114.000000,135.000000,180.000000,167.000000,58.000000,50.000000,70.000000,98.000000,231.000000,257.000000
6,7,335.0,144.000000,133.000000,133.000000,229.000000,234.000000,79.485714,50.000000,62.000000,94.000000,245.000000,217.000000
7,8,342.0,157.000000,119.000000,127.000000,196.000000,146.617647,79.485714,49.000000,72.000000,111.000000,240.000000,264.000000
8,9,351.0,112.000000,122.000000,152.000000,159.000000,149.000000,83.000000,48.000000,86.000000,117.000000,228.000000,162.000000
9,10,302.0,250.000000,140.000000,161.000000,154.000000,133.000000,103.000000,55.000000,107.000000,115.000000,204.000000,225.000000
